# Project 15 — Latent dimensions (probabilistic PCA / factor model)

**Scenario.** A panel of $D$ correlated measurements per sample (spectral channels, or omics features) is driven by a few underlying **latent factors**. We want to recover that low-dimensional structure.

**New skill.** Latent *continuous* structure (factor scores $z_n$). **Key pitfall.** *Rotational / sign non-identifiability* — the loadings $W$ and scores $z$ are identified only up to an orthogonal rotation, so raw $W$ entries are meaningless. We interpret **rotation-invariant** quantities (the noise $\sigma$, the reconstructed covariance $WW^\top+\sigma^2 I$) instead.

In [ ]:
import sys, pathlib
sys.path.insert(0, r'/home/user/biofx_python/bayesian_workflow_portfolio')
sys.path.insert(0, str(pathlib.Path.cwd()))
import warnings; warnings.filterwarnings('ignore')

In [ ]:
import numpy as np
import pymc as pm
import arviz as az
import matplotlib.pyplot as plt
az.style.use('arviz-darkgrid')
RNG = 20240601

## Step 1 — Problem & data-generating story

Probabilistic PCA: $z_n\sim\mathcal N(0,I_K)$, $x_n\sim\mathcal N(Wz_n+\mu,\sigma^2 I_D)$. **Assumptions:** (a) linear factor structure, (b) isotropic Gaussian noise (one shared $\sigma$), (c) a fixed number of factors $K$ (here the true $K=2$; over-specifying $K$ is the pitfall). Truth: $D=6, K=2, N=150, \sigma=0.4$.

In [ ]:
from data.generate_data import generate
data = generate(); X = data['X']; t = data['truth']
print(f"X shape = {X.shape}; true sigma={t['sigma']}, total_var(trace C)={t['total_var']:.2f}")

In [ ]:
fig, ax = plt.subplots(figsize=(5,4))
im = ax.imshow(np.corrcoef(X.T), cmap='RdBu_r', vmin=-1, vmax=1)
ax.set_title('Empirical correlation across the D dims\n(structure = shared factors)')
plt.colorbar(im, ax=ax, shrink=0.8); plt.tight_layout()

## Step 2 — Model specification (priors)

$$W\sim\mathcal N(0,1)^{D\times K},\quad \mu\sim\mathcal N(0,1)^D,\quad \sigma\sim\text{HalfNormal}(1),\quad z\sim\mathcal N(0,1)^{N\times K}.$$

**The identifiability caveat up front.** Because $W\to WR,\ z\to R^\top z$ (orthogonal $R$) leaves the likelihood unchanged, the posterior over raw $W$ is **not** unimodal and its R-hat will be large. This is expected and not a bug. We will only ever interpret rotation-invariant functions of $W$.

In [ ]:
from model import build_model, fit, reconstructed_cov, add_identifiable
model = build_model(data, K=2)
model

## Step 3 — Prior predictive checks

Datasets implied by the prior should have a covariance scale comparable to (or larger than) what we observe — the $\mathcal N(0,1)$ loadings give each dim unit-ish factor variance plus noise. We check the implied total variance is sensible (not collapsed to noise, not exploding).

In [ ]:
with model:
    prior = pm.sample_prior_predictive(draws=200, random_seed=RNG)
Xp = prior.prior_predictive['X'].values.reshape(-1, *X.shape)
tv = [np.trace(np.cov(xx.T)) for xx in Xp[:50]]
fig, ax = plt.subplots(figsize=(6,3.2))
ax.hist(tv, bins=20, color='#55A868', edgecolor='white')
ax.axvline(np.trace(np.cov(X.T)), color='red', label='observed total var')
ax.legend(); ax.set(xlabel='prior-implied total variance', title='Prior predictive')
plt.tight_layout()

## Step 4 — Inference (NUTS)

`draws=500, tune=1000, chains=2, target_accept=0.9`. The $N\times K$ latent scores make this a few-hundred-parameter model; it samples in ~30 s. Expect **clean** sampling for $\sigma$ and the reconstruction, but **poor** mixing for raw $W$/$z$ — by design.

In [ ]:
idata = fit(data, K=2, draws=500, tune=1000, chains=2, seed=15)
add_identifiable(idata)

## Step 5 — Diagnostics: read the right R-hat

Look at R-hat for **$\sigma$** and **total_var** (should be ≈ 1.0–1.05), NOT for raw $W$ (which will be large — the rotation symmetry, not a convergence failure). This is the central teaching point: *which* parameters you check depends on what is identified.

In [ ]:
print(az.summary(idata, var_names=['sigma','total_var']))
print('--- raw W R-hat is large ON PURPOSE (rotation non-identifiability): ---')
print(az.summary(idata, var_names=['W']).iloc[:4][['mean','r_hat','ess_bulk']])
print('divergences:', int(idata.sample_stats['diverging'].sum()))

## Step 6 — Posterior predictive checks

Compare the **observed covariance** to posterior-predictive covariances. The reconstruction $WW^\top+\sigma^2 I$ is rotation-invariant, so even though $W$ wanders, the implied covariance is stable and should match the data.

In [ ]:
C_hat = reconstructed_cov(idata)
C_emp = np.cov(X.T)
fig, axes = plt.subplots(1, 3, figsize=(11,3.2))
for ax, M, ttl in zip(axes, [C_emp, C_hat, data['C_true']],
                      ['empirical cov','reconstructed WWᵀ+σ²I','true C']):
    im = ax.imshow(M, cmap='viridis'); ax.set_title(ttl); plt.colorbar(im, ax=ax, shrink=0.7)
plt.tight_layout()
print('reconstruction rel Frobenius error =',
      round(np.linalg.norm(C_hat-data['C_true'])/np.linalg.norm(data['C_true']),3))

## Step 7 — Model criticism: how many factors?

Over-specifying $K$ is the analogue of the mixture's extra components: surplus factors are non-identified, hurt mixing, and do not improve fit. The robust check is the reconstruction error and (optionally) LOO across $K$. We compare the recovered $\sigma$ / total_var to truth as the identifiable recovery test.

In [ ]:
for name, truth in [('sigma', t['sigma']), ('total_var', t['total_var'])]:
    post = idata.posterior[name].values.ravel()
    lo, hi = np.percentile(post, [3, 97])
    print(f'{name:>10}: post mean={post.mean():.3f} 94%=[{lo:.3f},{hi:.3f}] truth={truth:.3f}')

## Step 8 — Decision & communication

For a collaborator: 'Two latent factors explain the bulk of the variance across the $D$ channels; the per-channel noise is $\sigma\approx0.4$. The factor *directions* are only defined up to rotation — interpret the reconstructed covariance and the dimensionality, not individual loading numbers.' See `summary_onepager.md`.